# Route Resilience — Graph Analysis & Simulation

Demonstrates the full graph pipeline on the OSM fallback:
1. Load OSM graph for Bengaluru AOI
2. Betweenness centrality → gatekeeper identification
3. Node ablation → Resilience Index
4. Cascading failure simulation
5. Emergency route comparison

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import networkx as nx
import osmnx as ox
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Download / load cached OSM graph
CACHE = '../data/graphs/osm_fallback.gpickle'
if os.path.exists(CACHE):
    import pickle
    with open(CACHE, 'rb') as f:
        G = pickle.load(f)
    print(f'Loaded from cache: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
else:
    G_osm = ox.graph_from_bbox(north=12.99, south=12.92, east=77.64, west=77.57, network_type='drive', simplify=True)
    G = ox.convert.to_undirected(G_osm)
    for n, d in G.nodes(data=True):
        d['x'] = d.get('x', d.get('lon', 0.0))
        d['y'] = d.get('y', d.get('lat', 0.0))
    for u, v, d in G.edges(data=True):
        d['weight'] = d.get('length', 1.0)
    print(f'Downloaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
from app.graph_pipeline.centrality import compute_betweenness
from app.graph_pipeline.metrics import compute_graph_metrics

metrics = compute_graph_metrics(G)
print('Graph metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

print('\nComputing betweenness centrality (k=200)...')
centrality = compute_betweenness(G, k=200)
top10 = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:10]
print('\nTop-10 gatekeeper nodes:')
for i, (node, score) in enumerate(top10):
    d = G.nodes[node]
    print(f'  #{i+1} node={node} centrality={score:.4f} lat={d.get("y",0):.5f} lon={d.get("x",0):.5f}')

In [ ]:
# Visualise centrality on map
fig, ax = plt.subplots(1, 1, figsize=(12, 10), facecolor='#0B0F1A')
ax.set_facecolor('#0B0F1A')

# Draw edges
for u, v in G.edges():
    xu, yu = G.nodes[u].get('x', 0), G.nodes[u].get('y', 0)
    xv, yv = G.nodes[v].get('x', 0), G.nodes[v].get('y', 0)
    ax.plot([xu, xv], [yu, yv], color='#1C2333', linewidth=0.5, zorder=1)

# Draw nodes coloured by centrality
cmap = plt.get_cmap('RdYlGn_r')
for node, score in centrality.items():
    d = G.nodes[node]
    x, y = d.get('x', 0), d.get('y', 0)
    size = 8 + score * 40
    ax.scatter(x, y, s=size, c=[score], cmap=cmap, vmin=0, vmax=1, zorder=2, alpha=0.9)

# Highlight top-5 gatekeepers
for i, (node, score) in enumerate(top10[:5]):
    d = G.nodes[node]
    ax.scatter(d.get('x', 0), d.get('y', 0), s=150, c='#FF4444', zorder=3, marker='*')
    ax.annotate(f'#{i+1}', (d.get('x', 0), d.get('y', 0)), color='white', fontsize=8, ha='center', va='bottom')

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
plt.colorbar(sm, ax=ax, label='Betweenness Centrality', fraction=0.03)
ax.set_title('Bengaluru Road Network — Criticality Heatmap', color='white', pad=12)
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('../data/criticality_heatmap.png', dpi=120, bbox_inches='tight', facecolor='#0B0F1A')
plt.show()

In [ ]:
# Ablation + Resilience Index
from app.simulation.ablation import ablate_nodes
from app.simulation.resilience import compute_resilience_index

top_nodes = [node for node, _ in top10[:3]]
print(f'Ablating top-3 gatekeeper nodes: {top_nodes}')

G_perturbed = ablate_nodes(G, top_nodes)
ri = compute_resilience_index(G, G_perturbed, sample_size=150)

print(f'\nResilience Index: {ri["resilience_index"]}')
print(f'Baseline avg path:  {ri["baseline_avg_path"]}')
print(f'Perturbed avg path: {ri["perturbed_avg_path"]}')
print(f'Disconnected: {ri["disconnected"]} ({ri["partition_count"]} partitions)')

In [ ]:
# Cascading failure
from app.simulation.cascade import run_cascade

steps = run_cascade(G, seed_nodes=[top_nodes[0]], max_iterations=3, threshold=0.6)
print('Cascade steps:')
for step in steps:
    print(f'  Iter {step["iteration"]}: ablated={len(step["ablated"])}, stressed={len(step["newly_stressed"])}, components={step["component_count"]}')